# Day 3 — Multimodal: Making Claude See Food

## What I learned today

- Images are binary, but HTTP/JSON is plain text — base64 encoding 
  bridges this at a 33% size cost
- Claude doesn't "see" pixels; a vision encoder converts images into 
  image tokens, processed identically alongside text tokens
- One 1024×1024 image ≈ 1600 tokens, equivalent to ~400 words of text 
  cost — image queries are 100-400× more expensive than typical text
- Sweet spot for recognition: 512–1024 px (below = accuracy drops, 
  above = diminishing returns + cost explodes)
- `content` becomes a list of typed blocks (`type: image` and 
  `type: text`); order matters — image first, instruction second

## Design choices I made + reasoning

- **Image source: base64** — NomNom photos are local user uploads, 
  not public URLs
- **Image size: compress to 1024 px longest edge** — sweet spot 
  (~1400 tokens vs ~16,000 for iPhone original, ~90% saving)
- **Block order: image first, text instruction second** — Anthropic 
  recommended, empirically better quality
- **Model: Haiku 4.5** for sandbox learning; Phase 4 will revisit 
  whether identification needs Sonnet

---

<Interview Q&A — Multimodal>

### Q1: Why can't pictures be directly placed into an API request? What does base64 encoding do? What's the cost?

Images are inherently binary data, but HTTP/JSON is a plain-text 
protocol — it can't carry binary directly. Base64 encoding converts 
the binary image bytes into an ASCII character string (like 
`"iVBORw0KGgo..."`), which JSON can carry. The cost is a **33% size 
increase**, because every 3 bytes of binary become 4 ASCII characters 
(base64 uses 64 distinct characters, so each character encodes 6 bits).

### Q2: How do you send an image to the API in Python?

Three-step pipeline expressed as one line:

```python
image_data = base64.standard_b64encode(path.read_bytes()).decode("utf-8")
```

- `read_bytes()`: load raw binary from disk
- `b64encode()`: convert binary → base64-encoded bytes
- `decode("utf-8")`: bytes → str (JSON-safe ASCII)

The resulting string is then placed in the message content as the 
`data` field of a source block.

### Q3: How does Claude handle images internally? Why are images so expensive?

Claude's core model is a Transformer — specifically a **decoder-only 
autoregressive** architecture. It only consumes a sequence of tokens 
(numbers), not raw pixels. To handle images, a separate **vision 
encoder** converts the image into image tokens (typically hundreds 
to thousands per image). These image tokens are concatenated with 
text tokens into a single sequence and processed **uniformly** by 
the model — Claude doesn't distinguish "image" from "text" at this 
layer.

**Cost implication**: a 1024×1024 image consumes ~1600 input tokens, 
roughly equivalent to a 400-word text input. For NomNom, every food 
photo costs as much as sending a short blog post — making image 
processing **100–400× more expensive** than typical text-only queries.

### Q4: Image size tradeoffs?

Four dimensions to balance when choosing image resolution:

1. **Accuracy** — too small → Claude can't distinguish details 
   (e.g., rice vs. oatmeal); too large → diminishing returns
2. **Cost** — token count scales as `tokens ≈ (width × height) / 750`. 
   Doubling resolution quadruples cost.
3. **Latency** — larger images take longer in the vision encoder, 
   delaying time-to-first-token
4. **API hard limits** (per Anthropic docs):
   - Max 5 MB per image (file size)
   - Single-image request: max 8000×8000 px
   - **Multi-image request: max 2000×2000 px per image**
   - Max 20 images per message; max 100 per request

**Token estimation reference**:

| Resolution | Approx tokens |
|---|---|
| 256×256 | ~87 |
| 512×512 | ~349 |
| **1024×1024** | **~1398** ← sweet spot |
| 2048×2048 | ~5592 |
| 3024×4032 (iPhone original) | ~16,257 |

**Sweet spot for recognition tasks**: 512–1024 px.
- Below 512 → accuracy drops on fine-grained categories
- Above 1024 → cost explodes with marginal accuracy gain

**NomNom strategy**: compress user-uploaded photos to 1024-px longest 
edge before API call (~1400 tokens vs ~16,000 for iPhone original) — 
a ~90% cost saving with no measurable accuracy loss. Phase 4 will 
revisit this in NomNom's cost dashboard.

### Q5: How are text and image combined in a request?

When sending a multimodal request, the `content` field changes from a 
plain string to a **list of blocks**, where each block is a dict with 
a `type` field indicating what it carries.

Two block types matter:
- **Image block**: `{"type": "image", "source": {...}}` — `source` can be 
  base64-encoded payload (local files, NomNom's case) or a public URL 
  (cheaper since you skip the 33% base64 overhead)
- **Text block**: `{"type": "text", "text": "..."}`

```python
"content": [
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/jpeg",
            "data": image_b64
        }
    },
    {
        "type": "text",
        "text": "What food is in this image? Estimate macros."
    }
]
```

**Order matters**: Anthropic recommends **image first, text 
instruction second**. Claude perceives the image, then receives the 
instruction about it — cognitively more natural and empirically 
produces better quality.

</Interview Q&A — Multimodal>

In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
client = Anthropic()

import base64

In [2]:
# define 3 helper functions
def add_user_message(messages, user_content):
    new_message = {
        "role": "user",
        "content": user_content
    }
    messages.append(new_message)


def chat(model, max_tokens, messages):
    response = client.messages.create(
        model = model,
        max_tokens = max_tokens,
        messages = messages
    )
    print("token usage: ", response.usage)
    print("stop reason: ", response.stop_reason)
    return response.content[0].text


In [3]:
# Encode a PNG image into 64 encoding
with open("food_test.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

In [4]:
# make a request including both image and text
user_content = [
    # image block
    {
        "type":"image",
        "source":{
            "type": "base64",
            "media_type": "image/png",  
            "data": image_bytes
        }
    },
    #text block
    {"type": "text",
     "text": "What do you see in this image?"
    }
]

model = "claude-haiku-4-5-20251001"
max_tokens = 5000
messages = []

add_user_message(messages, user_content)
answer = chat(model, max_tokens, messages)

token usage:  Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=1555, output_tokens=264, server_tool_use=None, service_tier='standard')
stop reason:  end_turn


In [5]:
answer

'# Buddha Bowl with Fresh Ingredients\n\nThis image shows a beautifully arranged **Buddha bowl** (or grain/salad bowl) on a rustic wooden surface. The plate contains a colorful variety of fresh, healthy ingredients arranged in sections:\n\n## Key Components:\n\n- **Leafy Greens**: Fresh lettuce and microgreens as the base\n- **Vegetables**:\n  - Cherry tomatoes on the vine (red)\n  - Yellow bell pepper\n  - Sliced avocado (green)\n  - Watermelon radish (pink/magenta with white center)\n  - Purple cabbage (shredded)\n  - Roasted sweet potato cubes (orange)\n\n- **Protein Source**: Chickpeas (garbanzo beans) - a plant-based protein\n\n- **Dressing/Garnish**: White creamy dressing drizzled over portions\n\nThe bowl is artfully presented with ingredients arranged in a circular, rainbow pattern, emphasizing the "whole foods" and plant-based nature of the meal. This is a typical example of a nutritious, Instagram-worthy healthy eating option that\'s both visually appealing and nutrient-dense